In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
import gc

# 1. Loading 'train_engineered.csv'
print("Loading Engineered data...")

df = pd.read_csv('../data/processed/train_engineered.csv')

# Dropping Transaction ID
X = df.drop(columns=['isFraud', 'TransactionID'], errors='ignore')
y = df['isFraud']

# 2. Downcasting to save memory space
print(" Applying Memory Optimizations (Downcasting)")
X = X.select_dtypes(include = ['number', 'bool'])
X = X.astype(np.float32)

# 3. Sampling data for Grid Search
print("Sample for Grid Search")
_, X_search, _, y_search = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

# 4. Garbage Collection
del df, X, y
gc.collect()
print("Optimized Grid Search Subset Shape: {X_search.shape}")

# 5. Class imbalance and weight setting
print("Calculating Class Imbalance...")
ratio = float(y_search.value_counts()[0] / y_search.value_counts()[1])
print(f"Scale Pos Weight configured to: {ratio:.2f}")

# 6. Grid SearchCV
print("Configuring GridSearchCV...")
base_model = xgb.XGBClassifier(
    tree_method='hist',
    scale_pos_weight=ratio,
    random_state=42
)

param_grid = {
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 300],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='average_precision',
    cv=3,
    verbose=2,
    n_jobs=1
)

print("Executing GridSearchCV...")
grid_search.fit(X_search, y_search)

print(f"Best Mathematical Combination: {grid_search.best_params_}")
print(f"Best AUPRC Score: {grid_search.best_score_:.4f}")


Loading Engineered data...
 Applying Memory Optimizations (Downcasting)
Sample for Grid Search
Optimized Grid Search Subset Shape: {X_search.shape}
Calculating Class Imbalance...
Scale Pos Weight configured to: 27.52
Configuring GridSearchCV...
Executing GridSearchCV...
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   7.6s
[CV] END learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   5.8s
[CV] END learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   5.7s
[CV] END learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   5.8s
[CV] END learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   5.7s
[CV] END learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   5.5s
[CV] END learning_rate=0.05, max_depth=4, n_estimators=300, subsample=0.8; total time=  13.4s
[CV] END l